In [5]:
import numpy as np
import pandas as pd
import yfinance as yf
import math
from typing import Dict, List, Sequence, Tuple

Data

In [4]:
VOL_TARGET = 0.15
VOL_LOOKBACK = 60
LOOKBACK = 126
RETURN_HORIZONS = (1, 21, 63, 126, 252)
MACD_PAIRS = ((8, 24), (16, 48), (32, 96))
EPS = 1e-8

In [ ]:
def download_adjusted_close(
    tickers: Sequence[str],
    start: str,
    end: str,
) -> pd.DataFrame:

    data = yf.download(tickers, start=start, end=end, progress=False)['Adj Close']

    if isinstance(data, pd.Series):
        data = data.to_frame(name=tickers[0] if isinstance(tickers, list) else tickers)

    return data.sort_index().dropna(how="all")

In [8]:
def calc_returns(price: pd.Series, day_offset: int = 1) -> pd.Series:
    return price / price.shift(day_offset) - 1.0

In [7]:
def calc_daily_vol(daily_returns: pd.Series, span: int = VOL_LOOKBACK) -> pd.Series:
    return daily_returns.ewm(span=span, min_periods=span, adjust=False).std()

In [ ]:
def calc_vol_scaled_returns(
    daily_returns: pd.Series,
    daily_vol: pd.Series,
    vol_target: float = VOL_TARGET,
) -> pd.Series:
    annualised_vol = daily_vol * np.sqrt(252.0)
    return daily_returns * vol_target / annualised_vol.shift(1).clip(lower=EPS)

In [9]:
def _halflife_from_timescale(timescale: int) -> float:
    return np.log(0.5) / np.log(1.0 - 1.0 / timescale)

In [ ]:

def calc_macd_signal(close: pd.Series, short_window: int, long_window: int) -> pd.Series:
    """
    Momentum Transformer-style MACD feature.
    """
    short_ewm = close.ewm(halflife=_halflife_from_timescale(short_window), adjust=False).mean()
    long_ewm = close.ewm(halflife=_halflife_from_timescale(long_window), adjust=False).mean()

    macd = short_ewm - long_ewm
    q = macd / close.rolling(63, min_periods=63).std().clip(lower=EPS)
    return q / q.rolling(252, min_periods=252).std().clip(lower=EPS)

In [ ]:
def build_asset_features(
    close: pd.Series,
    ticker: str,
) -> pd.DataFrame:
    close = close.dropna()
    close = close[close > 1e-8].copy()

    df = pd.DataFrame(index=close.index)
    df["date"] = df.index
    df["ticker"] = ticker
    df["close"] = close

    # Raw daily return
    df["daily_return"] = calc_returns(close)

    # Ex-ante daily volatility
    df["daily_vol"] = calc_daily_vol(df["daily_return"])
    
    # volatility-scaled same-day return as the target, and the next day's raw return as a sanity check.
    scaled_same_day = calc_vol_scaled_returns(df["daily_return"], df["daily_vol"])
    df["target_return"] = scaled_same_day.shift(-1)
    df["next_return"] = df["daily_return"].shift(-1)

    # Normalized multi-horizon returns
    for h in RETURN_HORIZONS:
        df[f"norm_ret_{h}"] = (
            calc_returns(close, h) / df["daily_vol"].clip(lower=EPS) / np.sqrt(h)
        )

    # MACD features
    for short_window, long_window in MACD_PAIRS:
        df[f"macd_{short_window}_{long_window}"] = calc_macd_signal(
            close, short_window, long_window
        )

    feature_cols = [f"norm_ret_{h}" for h in RETURN_HORIZONS] + [
        f"macd_{s}_{l}" for s, l in MACD_PAIRS
    ]

    # Drop warm-up rows, do not backfill.
    df = df.dropna(subset=["target_return", "daily_vol", *feature_cols]).copy()
    return df.reset_index(drop=True)

In [ ]:
def build_feature_panel(
    prices: pd.DataFrame,
    require_common_start: bool = True,
) -> Tuple[pd.DataFrame, List[str], Dict[str, int]]:
    """
    Combine all ETFs into one panel.
    """
    parts = []
    for ticker in prices.columns:
        parts.append(build_asset_features(prices[ticker], ticker))

    panel = pd.concat(parts, axis=0, ignore_index=True)
    panel["date"] = pd.to_datetime(panel["date"])

    # start when every ETF already exists, so later cross-sectional work is easier.
    if require_common_start:
        common_start = panel.groupby("ticker")["date"].min().max()
        panel = panel[panel["date"] >= common_start].copy()

    tickers = sorted(panel["ticker"].unique())
    ticker_to_id = {ticker: i for i, ticker in enumerate(tickers)}
    panel["asset_id"] = panel["ticker"].map(ticker_to_id).astype(np.int64)

    feature_cols = [f"norm_ret_{h}" for h in RETURN_HORIZONS] + [
        f"macd_{s}_{l}" for s, l in MACD_PAIRS
    ]

    panel = panel.sort_values(["ticker", "date"]).reset_index(drop=True)
    return panel, feature_cols, ticker_to_id